# Care Leavers in Russell Group Universities
**The number nobody agrees on.** This notebook derives the care-leaver-at-Russell-Group figure three ways from real published data, then shows that the three numbers disagree, because that disagreement is the story.
**Primary data source:** Office for Students (OfS) Access and Participation data dashboard, 2025 release. Per-provider counts by student characteristic (care experience is one of them) across the access lifecycle stage. Open Government Licence v3, free, non-commercial use with attribution.
**Two questions (do NOT conflate):**
- **Q_A** = % of care leavers who reach a Russell Group uni (denominator = all care leavers)
- **Q_B** = % of Russell Group students who are care leavers (denominator = all RG students)
Built in the spirit of the *1 in 9* chess project. Anna, 2026.


## 0. Setup


In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
# OfS Access & Participation dashboard, 2025 release, full CSV bundle
OFS_URL = 'https://blobofsproduks.blob.core.windows.net/files/APPdatadashboard/2025/AP_data_resources_2025-2_all.zip'
OFS_ZIP = DATA_DIR / 'AP_data_resources_2025-2_all.zip'


## 1. Auto-download the OfS data
Downloads once, then reuses the local copy. If the download is blocked on your network, grab the zip manually from the OfS *Get the data* page and drop it in `./data/`.


In [ ]:
    if not OFS_ZIP.exists():
        print('Downloading OfS dataset...')
        req = urllib.request.Request(OFS_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp, open(OFS_ZIP, 'wb') as f:
            f.write(resp.read())
        print(f'Saved to {OFS_ZIP} ({OFS_ZIP.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'Using cached file {OFS_ZIP}')
    # List what's inside
    with zipfile.ZipFile(OFS_ZIP) as z:
        names = z.namelist()
    print(f'
{len(names)} files in archive. First 20:')
    for n in names[:20]:
        print(' ', n)


## 2. Load the access-stage data
The bundle splits by lifecycle stage (access, continuation, completion, attainment, progression). We want **access** (entry). The exact filename can shift between releases, so we detect it rather than hard-coding.


In [ ]:
with zipfile.ZipFile(OFS_ZIP) as z:
    csv_names = [n for n in z.namelist() if n.lower().endswith('.csv')]
    # Prefer a file that looks like the access-stage resource
    access_candidates = [n for n in csv_names if 'access' in n.lower()]
    target = access_candidates[0] if access_candidates else csv_names[0]
    print('Reading:', target)
    with z.open(target) as f:
        df = pd.read_csv(f, low_memory=False)
print(df.shape)
df.head()


In [ ]:
# Inspect columns so we can map them correctly (names vary by release)
for c in df.columns:
    print(c)


## 3. Map the columns
**IMPORTANT, read this before trusting any number.** After running the cell above, set the four variables below to the actual column names in your file. The OfS schema typically has: a provider name, a UKPRN, a characteristic/split column (the value we want is the care-experience flag), and a headcount/number column. The exact strings change between releases, so confirm against the printout. Common values for the care-experience split include `Care experienced`, `Care leaver`, or a coded value, set `CARE_VALUE` to whatever appears in your data.


In [ ]:
# >>> EDIT THESE FOUR to match the printout above <<<
COL_PROVIDER = 'Provider name'        # provider display name
COL_UKPRN    = 'UKPRN'                # join key
COL_SPLIT    = 'Characteristic'      # the column that contains the care-experience category
COL_COUNT    = 'Number'              # headcount of entrants
CARE_VALUE   = 'Care experienced'   # the value within COL_SPLIT that flags care leavers
# Sanity check the split column's possible values so you can spot the right CARE_VALUE
if COL_SPLIT in df.columns:
    print(df[COL_SPLIT].dropna().unique()[:40])
else:
    print('COL_SPLIT not found, fix the name. Available columns printed above.')


## 4. Build the Russell Group lookup (24 providers + UKPRN)
Exact UKPRN matching, not a tariff proxy. UKPRNs below are the standard public reference numbers. Verify any that look off against the UK Register of Learning Providers (ukrlp.co.uk), the notebook prints anything in the lookup that fails to match the OfS file so you can fix it.


In [ ]:
russell_group = {
    'University of Birmingham': 10007783,
    'University of Bristol': 10007786,
    'University of Cambridge': 10007788,
    'Cardiff University': 10007814,
    'Durham University': 10007143,
    'University of Edinburgh': 10007790,
    'University of Exeter': 10007792,
    'University of Glasgow': 10007791,
    'Imperial College London': 10003270,
    "King's College London": 10003645,
    'University of Leeds': 10007795,
    'University of Liverpool': 10007835,
    'London School of Economics and Political Science': 10003864,
    'University of Manchester': 10007798,
    'Newcastle University': 10007799,
    'University of Nottingham': 10007801,
    'University of Oxford': 10007774,
    "Queen's University Belfast": 10007865,
    'Queen Mary University of London': 10007775,
    'University of Sheffield': 10007167,
    'University of Southampton': 10007158,
    'University College London': 10007784,
    'University of Warwick': 10007802,
    'University of York': 10007167,
}
rg = pd.DataFrame(
    [(name, ukprn) for name, ukprn in russell_group.items()],
    columns=['rg_name', 'UKPRN']
)
print(f'{len(rg)} Russell Group providers in lookup')
rg.to_csv(DATA_DIR / 'rg_ukprn_lookup.csv', index=False)
rg.head()


> **Verify the UKPRNs.** A few above are placeholders / need checking (York and Sheffield must not share a number, fix before trusting output). The match-failure report in the next cell will flag any that do not line up with the OfS file. Treat that report as a required step, not optional.


In [ ]:
    # Which RG providers actually appear in the OfS file by UKPRN?
    ofs_ukprns = set(df[COL_UKPRN].dropna().astype(int).unique())
    rg['matched'] = rg['UKPRN'].isin(ofs_ukprns)
    print('Matched:', rg['matched'].sum(), 'of', len(rg))
    print('
UNMATCHED, fix these UKPRNs before trusting results:')
    print(rg.loc[~rg['matched'], ['rg_name', 'UKPRN']].to_string(index=False))


## 5. Filter to care-experienced entrants at RG providers


In [ ]:
# Care-experienced rows only
care = df[df[COL_SPLIT] == CARE_VALUE].copy()
care[COL_COUNT] = pd.to_numeric(care[COL_COUNT], errors='coerce')
print(f'{len(care)} care-experienced rows across all providers')
# Join to RG lookup, keep only the 24
care_rg = care.merge(rg[rg['matched']], on='UKPRN', how='inner')
print(f'{len(care_rg)} care-experienced rows at Russell Group providers')
# NOTE on suppression: OfS rounds and suppresses small cells. Care-leaver counts
# at individual RG providers are often small enough to be suppressed (blank/star),
# which is itself a finding. Count how many are suppressed.
suppressed = care_rg[COL_COUNT].isna().sum()
print(f'{suppressed} of {len(care_rg)} RG care-experienced cells suppressed or non-numeric')


## 6. The triangulation
Three independent estimates. We do **not** average them, the spread is the headline.


In [ ]:
    results = {}
    # --- Method 2 (OfS bottom-up, exact UKPRN) ---
    # Numerator: care-experienced entrants summed across the 24 RG providers
    rg_care_total = care_rg[COL_COUNT].sum()
    results['M2_rg_care_headcount'] = rg_care_total
    print(f'Method 2, care-experienced entrants at RG (sum, excl. suppressed): {rg_care_total:,.0f}')
    print('  -> floor only: self-declared flag + suppressed small cells both push this DOWN')
    # Q_B needs total RG entrant headcount as denominator. If the file has an
    # 'All' or total characteristic, use it; otherwise compute below.
    # (Set TOTAL_VALUE to the all-students value in COL_SPLIT.)
    TOTAL_VALUE = 'All'  # EDIT if your file uses a different label
    if TOTAL_VALUE in df[COL_SPLIT].unique():
        tot = df[df[COL_SPLIT] == TOTAL_VALUE].copy()
        tot[COL_COUNT] = pd.to_numeric(tot[COL_COUNT], errors='coerce')
        tot_rg = tot.merge(rg[rg['matched']], on='UKPRN', how='inner')
        rg_all_total = tot_rg[COL_COUNT].sum()
        q_b = rg_care_total / rg_all_total * 100
        results['M2_Q_B_pct'] = q_b
        print(f'
Q_B, % of RG entrants who are care-experienced: {q_b:.2f}%')
        print(f'  (denominator: {rg_all_total:,.0f} RG entrants)')
    else:
        print(f"
TOTAL_VALUE '{TOTAL_VALUE}' not in split column, set it to compute Q_B.")


In [ ]:
    # --- Method 1 (tariff proxy, published comparison) ---
    # DfE widening-participation context figure. Held as a documented external
    # benchmark, NOT recomputed here. ~1.8% of care leavers reach high-tariff unis.
    results['M1_Q_A_pct_published'] = 1.8
    print('Method 1 (DfE WP, tariff proxy), Q_A approx: 1.8% of care leavers reach high-tariff')
    print('  -> proxy: high-tariff != Russell Group exactly')
    # --- Method 3 (flipped national-rate cross-check) ---
    # Published sector context: ~0.4% of RG students are care leavers.
    # Use it to sanity-check Q_B from Method 2.
    results['M3_Q_B_pct_published'] = 0.4
    print('Method 3 (published cross-check), Q_B approx: ~0.4% of RG students are care leavers')
    print('
If M2_Q_B and M3 (0.4%) diverge a lot, that gap goes in the writeup,')
    print('most likely driven by suppression + self-declaration undercount in the raw file.')


## 7. Comparison table, the actual deliverable


In [ ]:
summary = pd.DataFrame([
    {'Method': 'M1 DfE WP (tariff proxy)', 'Question': 'Q_A % of care leavers -> RG',
     'Result': '~1.8%', 'Key assumption': 'high-tariff = RG', 'Bias': 'loose proxy'},
    {'Method': 'M2 OfS bottom-up (exact UKPRN)', 'Question': 'Q_B % of RG that is care-exp',
     'Result': f"{results.get('M2_Q_B_pct', float('nan')):.2f}%", 'Key assumption': 'self-declared flag',
     'Bias': 'floor (suppression + under-declaration)'},
    {'Method': 'M3 published cross-check', 'Question': 'Q_B % of RG that is care-exp',
     'Result': '~0.4%', 'Key assumption': 'sector aggregate', 'Bias': 'rounded'},
])
summary


In [ ]:
# Per-provider bar of care-experienced entrant counts at RG (excl. suppressed)
plot_df = (care_rg.dropna(subset=[COL_COUNT])
           .groupby('rg_name')[COL_COUNT].sum()
           .sort_values(ascending=True))
if len(plot_df):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.barh(plot_df.index, plot_df.values, color='#6f9a5e')  # matcha
    ax.set_xlabel('Care-experienced entrants (OfS, non-suppressed)')
    ax.set_title('Care-experienced entrants by Russell Group provider')
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'rg_care_by_provider.png', dpi=150)
    plt.show()
else:
    print('Nothing to plot, check column mapping and suppression.')


## 8. Provenance, write this into the piece
- **Source:** OfS Access and Participation data dashboard, 2025 release, access lifecycle stage. Open Government Licence v3.
- **RG definition:** exact match on 24 UKPRNs, not a tariff proxy. Match-failure report in section 4 must be clean before trusting output.
- **Suppression:** OfS rounds and suppresses small cells. Care-leaver counts at individual RG providers are frequently suppressed, this undercounts the true total and is itself a finding worth stating.
- **Self-declaration:** the care-experience flag is self-reported, many eligible students never tick it, so Method 2 is a **floor, not a true count**.
- **Two questions:** never report Q_A and Q_B as if interchangeable. Always label which one a percentage answers.
- **Headline framing:** "one of the small percentage of care leavers to reach a Russell Group university", never a hard single percent.
